# Download InfluxDB Raw Data
This notebook downloads one day of raw time-series data (`channel_metrics`) from InfluxDB and archives it to Google Drive as a CSV file.

- **Storage Location**: `/content/drive/MyDrive/Predikto/influxdb_raw/`
- **Partitioning**: Split by day (UTC) as `YYYY-MM-DD.csv`
- **Logic**: It checks the drive for existing files and downloads the earliest missing day, starting from yesterday and going backwards. It never downloads the present day to ensure the data for that day is complete.


In [ ]:
# Install required packages
!pip install influxdb-client pandas

In [ ]:
import os
import datetime
import pandas as pd
from google.colab import drive
from google.colab import userdata
from influxdb_client import InfluxDBClient

# Mount Google Drive
drive.mount('/content/drive')

# Setup InfluxDB Client
url = userdata.get('INFLUXDB_URL')
token = userdata.get('INFLUXDB_TOKEN')
org = userdata.get('INFLUXDB_ORG')
bucket = userdata.get('INFLUXDB_BUCKET')

if not all([url, token, org, bucket]):
    print("Warning: Missing InfluxDB credentials in Colab Secrets.")
else:
    client = InfluxDBClient(url=url, token=token, org=org, timeout=120_000)
    query_api = client.query_api()
    print("InfluxDB Client Initialized")


In [ ]:
# Define configuration
DRIVE_DIR = '/content/drive/MyDrive/Predikto/influxdb_raw/'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Helper function to format time for Flux
def format_flux_time(dt):
    return dt.isoformat().replace('+00:00', 'Z')

# Determine which day to query
today = datetime.datetime.now(datetime.timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
target_day = today - datetime.timedelta(days=1)

MAX_DAYS_BACK = 365
found_day_to_download = False

# Find the first past day that hasn't been downloaded yet
for _ in range(MAX_DAYS_BACK):
    file_name = f"{target_day.strftime('%Y-%m-%d')}.csv"
    file_path = os.path.join(DRIVE_DIR, file_name)
    
    if not os.path.exists(file_path):
        found_day_to_download = True
        break
    else:
        # Move one day back
        target_day = target_day - datetime.timedelta(days=1)

if not found_day_to_download:
    print("All past days up to the limit have been downloaded.")
else:
    print(f"Target day to download: {target_day.strftime('%Y-%m-%d')}")
    
    start_time = target_day
    end_time = target_day + datetime.timedelta(days=1)
    
    print(f"Querying InfluxDB from {start_time} to {end_time}...")
    
    # Flux query to get channel_metrics and pivot
    flux_query = f'''
        from(bucket: "{bucket}")
            |> range(start: {format_flux_time(start_time)}, stop: {format_flux_time(end_time)})
            |> filter(fn: (r) => r["_measurement"] == "channel_metrics")
            |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
    '''
    
    try:
        df = query_api.query_data_frame(flux_query)
        
        if isinstance(df, list):
            if not df:
                df = pd.DataFrame()
            else:
                df = pd.concat(df, ignore_index=True)
                
        if not df.empty:
            # Clean up columns if needed
            if 'result' in df.columns:
                df = df.drop(columns=['result'])
            if 'table' in df.columns:
                df = df.drop(columns=['table'])
            
            # Save to CSV
            df.to_csv(file_path, index=False)
            print(f"Successfully downloaded and saved {len(df)} rows to {file_path}")
        else:
            print(f"No data found for {target_day.strftime('%Y-%m-%d')}. Creating empty marker file.")
            # Create an empty file to avoid querying this day again
            pd.DataFrame().to_csv(file_path, index=False)
            
    except Exception as e:
        print(f"Error querying InfluxDB: {e}")
